# RoadNetra AI · Model B — Severe Accident Detection

Trains `accident_best.pt` for the 30 FPS emergency stream (rollovers, high-speed crashes, vehicle fires),
following *RoadNetra_ModelB_Accident_Requirements.pdf* and the Master Blueprint.

| Contract item | Value |
|---|---|
| Architecture | YOLOv8s (switch `MODEL_ARCH` to `yolov8n.pt` if CPU latency misses 75 ms) |
| Input | 640 × 640 RGB |
| Classes | `0: severe_accident` (single class) |
| Output | `accident_best.pt` → copy to `models/` |

**Dataset recipe** (all CC BY 4.0, Roboflow Universe)

| Role | Source | Use |
|---|---|---|
| Primary positives | [`tammy/cctv-accident`](https://universe.roboflow.com/tammy/cctv-accident) — 1,491 CCTV accident images | every image with an accident-type box |
| Fire supplement | [`yolovideos/accident-detection-bcc2v`](https://universe.roboflow.com/yolovideos/accident-detection-bcc2v) — 2,179 images | only images that contain a fire box |
| Negative control | [`atledtech/cctv_vehicles`](https://universe.roboflow.com/atledtech/cctv_vehicles) — 2,788 CCTV traffic images | 250 frames, labels emptied (traffic ≠ accident) |

Accident / crash / collision / rollover / overturn / fire classes are all merged into class 0; vehicle, person,
"moderate" and numeric junk classes are dropped. Positives are de-duplicated across sources and capped at 2,000.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then add your free Roboflow API key in the 🔑 *Secrets*
panel as `ROBOFLOW_API_KEY` (roboflow.com → Settings → API Keys). Then `Runtime → Run all` (~45–60 min).

In [ ]:
# 1 · Environment
!nvidia-smi -L
!pip install -q "ultralytics>=8.1.0" roboflow albumentations
import torch, ultralytics
print('torch', torch.__version__, '| ultralytics', ultralytics.__version__)
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2 · Configuration
import os
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    from getpass import getpass
    ROBOFLOW_API_KEY = getpass('Roboflow API key: ')

POSITIVE_SOURCES = [
    # (workspace, project, mode)   mode 'all': every image with an accident-type box
    #                              mode 'fire_only': only images that contain a fire box
    ('tammy', 'cctv-accident', 'all'),
    ('yolovideos', 'accident-detection-bcc2v', 'fire_only'),
]
NEGATIVE_SOURCE = ('atledtech', 'cctv_vehicles')

MAX_POSITIVES    = 2000   # hard cap on accident images (target band 1,500–2,000)
N_NEGATIVES      = 250    # intact-traffic frames with empty labels
WEATHER_FRACTION = 0.25   # share of train positives that get a rain/fog/blur/glare copy
MODEL_ARCH       = 'yolov8s.pt'
EPOCHS           = 40
SEED             = 0

ROOT = '/content/accident_dataset'
RAW  = '/content/raw'

# Persist runs to Google Drive so a Colab disconnect doesn't lose training
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/roadnetra_accident'
else:
    PROJECT = '/content/roadnetra_accident'
RUN = 'yolov8s_wild_v1'

In [ ]:
# 3 · Dataset builder (download → filter classes → merge → dedupe → negatives → weather copies)
import glob, hashlib, random, re, shutil
from collections import Counter
from pathlib import Path
import cv2, numpy as np, yaml

KEEP = re.compile(r'accident|crash|collision|collide|wreck|severe|fire|flame|burn|roll|overturn|flip|upside', re.I)
DROP = re.compile(r'non[-_ ]?accident|no[-_ ]?accident|not[-_ ]?accident|normal', re.I)
FIRE = re.compile(r'fire|flame|burn', re.I)
SPLITS = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')


def read_names(ds_dir):
    names = yaml.safe_load(open(Path(ds_dir) / 'data.yaml'))['names']
    return dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}


def iter_images(ds_dir):
    """Yields (split, image_path, label_path) for a Roboflow YOLO export."""
    for raw_split, split in SPLITS.items():
        img_dir = Path(ds_dir) / raw_split / 'images'
        if not img_dir.is_dir():
            continue
        for img in sorted(img_dir.iterdir()):
            if img.suffix.lower() in IMG_EXT:
                yield split, img, Path(ds_dir) / raw_split / 'labels' / (img.stem + '.txt')


def to_box(vals):
    """YOLO box line, or polygon (segmentation export) collapsed to its bounding box."""
    if len(vals) == 4:
        return vals
    xs, ys = vals[0::2], vals[1::2]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    return [(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0]


def filter_labels(label_path, names, mode, stats):
    """Returns remapped class-0 lines, or None if the image should be skipped."""
    if not label_path.exists():
        return None
    kept, has_fire, local = [], False, Counter()
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        name = str(names.get(int(float(parts[0])), parts[0]))
        if KEEP.search(name) and not DROP.search(name):
            local[name] += 1
            has_fire |= bool(FIRE.search(name))
            x, y, w, h = to_box([float(v) for v in parts[1:]])
            if w > 0 and h > 0:
                kept.append(f'0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
        else:
            local[f'(dropped) {name}'] += 1
    if not kept or (mode == 'fire_only' and not has_fire):
        return None
    stats.update(local)
    return kept


def ahash(img_path):
    g = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if g is None:
        return None
    g = cv2.resize(g, (16, 16), interpolation=cv2.INTER_AREA)
    return hashlib.md5((g > g.mean()).tobytes()).hexdigest()


def collect_positives(sources, max_positives, seed):
    """sources: list of (tag, ds_dir, mode). Returns list of dicts and per-class stats."""
    items, seen, stats = [], set(), Counter()
    dupes = 0
    for tag, ds_dir, mode in sources:
        names = read_names(ds_dir)
        print(f'{tag}: classes = {names}')
        n_before = len(items)
        for split, img, lbl in iter_images(ds_dir):
            lines = filter_labels(lbl, names, mode, stats)
            if lines is None:
                continue
            h = ahash(img)
            if h is None:
                continue
            if h in seen:
                dupes += 1
                continue
            seen.add(h)
            items.append(dict(src=tag, split=split, img=img, lines=lines))
        print(f'  → {len(items) - n_before} images kept')
    print(f'Cross-source/in-source duplicates removed: {dupes}')

    if len(items) > max_positives:
        # Trim primary-source train images first so the fire supplement and held-out splits survive
        rng = random.Random(seed)
        primary = sources[0][0]
        trimmable = [i for i, it in enumerate(items) if it['split'] == 'train' and it['src'] == primary]
        rng.shuffle(trimmable)
        drop = set(trimmable[:len(items) - max_positives])
        items = [it for i, it in enumerate(items) if i not in drop]
    return items, stats


def place(img_src, dst_root, split, name, lines):
    (Path(dst_root) / 'images' / split).mkdir(parents=True, exist_ok=True)
    (Path(dst_root) / 'labels' / split).mkdir(parents=True, exist_ok=True)
    shutil.copy(img_src, Path(dst_root) / 'images' / split / (name + Path(img_src).suffix.lower()))
    (Path(dst_root) / 'labels' / split / (name + '.txt')).write_text('\n'.join(lines) + ('\n' if lines else ''))


def add_negatives(neg_dir, dst_root, n, seed):
    pool = [img for _, img, _ in iter_images(neg_dir)]
    rng = random.Random(seed)
    picks = rng.sample(pool, min(n, len(pool)))
    n_val = max(1, len(picks) // 5)
    for i, img in enumerate(picks):
        place(img, dst_root, 'val' if i < n_val else 'train', f'neg_{i:04d}', [])
    return len(picks) - n_val, n_val


def add_weather_copies(dst_root, fraction, seed):
    """Pixel-level weather/blur/glare copies of train positives (boxes unchanged)."""
    try:
        import albumentations as A
        aug = A.OneOf([
            A.RandomRain(p=1.0),
            A.RandomFog(p=1.0),
            A.MotionBlur(blur_limit=(3, 9), p=1.0),
            A.GaussianBlur(p=1.0),
            A.RandomSunFlare(p=1.0),
            A.RandomBrightnessContrast(brightness_limit=(-0.5, 0.2), contrast_limit=(-0.4, 0.1), p=1.0),
        ], p=1.0)
    except Exception as e:
        print('albumentations unavailable, skipping weather copies:', e)
        return 0
    train_imgs = [p for p in sorted((Path(dst_root) / 'images' / 'train').iterdir()) if not p.name.startswith('neg_')]
    rng = random.Random(seed)
    made = 0
    for img_path in rng.sample(train_imgs, int(len(train_imgs) * fraction)):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        try:
            out = aug(image=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))['image']
        except Exception:
            continue
        stem = 'wx_' + img_path.stem
        cv2.imwrite(str(img_path.parent / (stem + '.jpg')), cv2.cvtColor(out, cv2.COLOR_RGB2BGR))
        shutil.copy(Path(dst_root) / 'labels' / 'train' / (img_path.stem + '.txt'),
                    Path(dst_root) / 'labels' / 'train' / (stem + '.txt'))
        made += 1
    return made


def build_dataset(pos_sources, neg_dir, dst_root, max_positives, n_negatives, weather_fraction, seed):
    shutil.rmtree(dst_root, ignore_errors=True)
    items, stats = collect_positives(pos_sources, max_positives, seed)
    for i, it in enumerate(items):
        place(it['img'], dst_root, it['split'], f"{it['src']}_{i:05d}", it['lines'])
    neg_train, neg_val = add_negatives(neg_dir, dst_root, n_negatives, seed) if neg_dir else (0, 0)
    n_wx = add_weather_copies(dst_root, weather_fraction, seed)

    has_test = (Path(dst_root) / 'images' / 'test').is_dir()
    cfg = {'path': dst_root, 'train': 'images/train', 'val': 'images/val', 'names': {0: 'severe_accident'}}
    if has_test:
        cfg['test'] = 'images/test'
    yaml.safe_dump(cfg, open(Path(dst_root) / 'data.yaml', 'w'), sort_keys=False)

    print('\nBoxes per source class:')
    for k, v in sorted(stats.items(), key=lambda kv: -kv[1]):
        print(f'  {v:6d}  {k}')
    split_count = Counter(it['split'] for it in items)
    src_count = Counter(it['src'] for it in items)
    print(f'\nAccident images: {len(items)}  by split {dict(split_count)}  by source {dict(src_count)}')
    print(f'Negatives: {neg_train} train / {neg_val} val   Weather copies (train): {n_wx}')
    return Path(dst_root) / 'data.yaml', len(items)

In [ ]:
# 4 · Download from Roboflow and build the merged dataset
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

def download(ws, proj_name):
    proj = rf.workspace(ws).project(proj_name)
    version = None
    try:
        nums = [int(str(v.version).rstrip('/').split('/')[-1]) for v in proj.versions()]
        version = proj.version(max(nums)) if nums else None
    except Exception as e:
        print('  versions() lookup failed, probing:', e)
    if version is None:
        for n in range(30, 0, -1):
            try:
                version = proj.version(n); break
            except Exception:
                pass
    print(f'Downloading {ws}/{proj_name} v{version.version}')
    return version.download('yolov8', location=f'{RAW}/{ws}__{proj_name}', overwrite=True).location

pos = [(f'{ws}__{p}', download(ws, p), mode) for ws, p, mode in POSITIVE_SOURCES]
neg_dir = download(*NEGATIVE_SOURCE) if NEGATIVE_SOURCE else None

DATA_YAML, N_POS = build_dataset(pos, neg_dir, ROOT, MAX_POSITIVES, N_NEGATIVES, WEATHER_FRACTION, SEED)
print(open(DATA_YAML).read())
if N_POS < 1500:
    print(f'⚠ Only {N_POS} accident images after filtering — check the class table above; '
          'if a source used a class name the KEEP regex missed, extend KEEP and re-run this cell.')

In [ ]:
# 5 · Visual audit — eyeball that boxes really are crashes / rollovers / fires
import matplotlib.pyplot as plt
imgs = sorted(glob.glob(f'{ROOT}/images/train/*'))
rng = random.Random(SEED)
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
for ax, p in zip(axes.flat, rng.sample(imgs, 12)):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); H, W = im.shape[:2]
    for line in open(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt').read().split('\n'):
        if line.strip():
            _, x, y, w, h = map(float, line.split())
            cv2.rectangle(im, (int((x - w/2)*W), int((y - h/2)*H)), (int((x + w/2)*W), int((y + h/2)*H)), (255, 0, 0), 3)
    ax.imshow(im); ax.set_title(Path(p).name, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 6 · Train with 'Vision in the Wild' augmentations (spec §5.2)
from ultralytics import YOLO
model = YOLO(MODEL_ARCH)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,
    seed=SEED,
    # --- 'Vision in the Wild' augmentation hyperparameters ---
    hsv_h=0.015,         # hue
    hsv_s=0.7,           # saturation jitter (smog, heavy rain)
    hsv_v=0.4,           # brightness jitter (night CCTV, tunnels, glare)
    degrees=10.0,        # pole sway / camera tilt
    translate=0.1,       # frame shifts from vibration
    scale=0.5,           # near vs distant crashes
    perspective=0.0005,  # oblique high-mast angles
    mosaic=1.0,          # occlusion, multi-vehicle scenes
    mixup=0.15,          # background clutter
    # Rain / fog / motion blur / glare are baked in offline by the dataset builder (WEATHER_FRACTION)
    project=PROJECT,
    name=RUN,
    exist_ok=True,
)
# If Colab disconnects mid-training: YOLO(f'{PROJECT}/{RUN}/weights/last.pt').train(resume=True)

In [ ]:
# 7 · Evaluate: validation, held-out test, and false alarms on intact traffic
BEST = f'{PROJECT}/{RUN}/weights/best.pt'
best = YOLO(BEST)

m = best.val(data=str(DATA_YAML), split='val', imgsz=640, verbose=False)
print(f'VAL   mAP50 {m.box.map50:.4f} | mAP50-95 {m.box.map:.4f} | P {m.box.mp:.4f} | R {m.box.mr:.4f}')
if os.path.isdir(f'{ROOT}/images/test'):
    t = best.val(data=str(DATA_YAML), split='test', imgsz=640, verbose=False)
    print(f'TEST  mAP50 {t.box.map50:.4f} | mAP50-95 {t.box.map:.4f} | P {t.box.mp:.4f} | R {t.box.mr:.4f}')

# Operating point used by the RoadNetra temporal filter: conf >= 0.55
neg = sorted(glob.glob(f'{ROOT}/images/val/neg_*'))
fp = sum(len(r.boxes) > 0 for r in best.predict(neg, conf=0.55, imgsz=640, verbose=False, stream=True))
print(f'False alarms on intact traffic @conf 0.55: {fp}/{len(neg)} frames ({100*fp/max(1,len(neg)):.1f}%)')

In [ ]:
# 8 · Latency benchmark vs contract (≤25 ms GPU / ≤75 ms CPU)
import time
frame = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)  # 720p CCTV-sized frame

def bench(device, n):
    for _ in range(10):
        best.predict(frame, imgsz=640, device=device, verbose=False)
    ts = []
    for _ in range(n):
        s = time.perf_counter(); best.predict(frame, imgsz=640, device=device, verbose=False)
        ts.append((time.perf_counter() - s) * 1000)
    return np.median(ts), np.percentile(ts, 95)

g50, g95 = bench(0, 200)
c50, c95 = bench('cpu', 30)
print(f'GPU: median {g50:.1f} ms, p95 {g95:.1f} ms  → {"PASS" if g50 <= 25 else "FAIL"} (≤25 ms)')
print(f'CPU: median {c50:.1f} ms, p95 {c95:.1f} ms  → {"PASS" if c50 <= 75 else "FAIL"} (≤75 ms)')
print('Note: Colab CPU is 2 weak vCPUs — re-check on the edge machine with training/benchmark_local.py')

In [ ]:
# 9 · Contract check + export accident_best.pt
import shutil
assert best.names == {0: 'severe_accident'}, best.names
assert best.ckpt['train_args']['imgsz'] == 640
shutil.copy(BEST, '/content/accident_best.pt')
shutil.make_archive('/content/accident_run_artifacts', 'zip', f'{PROJECT}/{RUN}')  # curves, confusion matrix for the deck
print('Contract OK:', best.names, '| ultralytics', ultralytics.__version__, '| torch', torch.__version__)

from google.colab import files
files.download('/content/accident_best.pt')
files.download('/content/accident_run_artifacts.zip')

## Local handoff
1. Put `accident_best.pt` in `ROADNETRA-AI/models/`.
2. On the edge laptop: `python training/benchmark_local.py` — checks class map and CPU latency against the 75 ms target.
3. The dual-stream engine loads it with `YOLO('models/accident_best.pt')`; the temporal filter requires ≥3 consecutive frames at conf ≥ 0.55.